# Question Answering with LangChain, OpenAI, and MultiQuery Retriever

This interactive workbook demonstrates example of Elasticsearch's [MultiQuery Retriever](https://api.python.langchain.com/en/latest/retrievers/langchain.retrievers.multi_query.MultiQueryRetriever.html) to generate similar queries for a given user input and apply all queries to retrieve a larger set of relevant documents from a vectorstore.

Before we begin, we first split the fictional workplace documents into passages with `langchain` and uses OpenAI to transform these passages into embeddings and then store these into Elasticsearch.

We will then ask a question, generate similar questions using langchain and OpenAI, retrieve relevant passages from the vector store, and use langchain and OpenAI again to provide a summary for the questions.

## Install packages and import modules

In [3]:
!python3 -m pip install -qU jq lark langchain langchain-elasticsearch langchain_openai tiktoken

from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_elasticsearch import ElasticsearchStore
from langchain_openai.llms import OpenAI
from langchain.retrievers.multi_query import MultiQueryRetriever
from getpass import getpass


[notice] A new release of pip is available: 24.0 -> 25.2
[notice] To update, run: C:\Users\nicol\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


## Connect to Elasticsearch

ℹ️ We're using an Elastic Cloud deployment of Elasticsearch for this notebook. If you don't have an Elastic Cloud deployment, sign up [here](https://cloud.elastic.co/registration?utm_source=github&utm_content=elasticsearch-labs-notebook) for a free trial. 

We'll use the **Cloud ID** to identify our deployment, because we are using Elastic Cloud deployment. To find the Cloud ID for your deployment, go to https://cloud.elastic.co/deployments and select your deployment.

We will use [ElasticsearchStore](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html) to connect to our elastic cloud deployment, This would help create and index data easily.  We would also send list of documents that we created in the previous step

In [9]:
# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#finding-your-cloud-id
ELASTIC_CLOUD_ID = getpass("Elastic Cloud ID: ")

# https://www.elastic.co/search-labs/tutorials/install-elasticsearch/elastic-cloud#creating-an-api-key
ELASTIC_API_KEY = getpass("Elastic Api Key: ")

# https://platform.openai.com/api-keys
OPENAI_API_KEY = getpass("OpenAI API key: ")

embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

In [10]:
vectorstore = ElasticsearchStore(
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
    index_name= "langchain_multiquery", #give it a meaningful name,
    embedding=embeddings,
)

## Indexing Data into Elasticsearch
Let's download the sample dataset and deserialize the document.

In [11]:
from urllib.request import urlopen
import json

url = "https://raw.githubusercontent.com/elastic/elasticsearch-labs/main/example-apps/chatbot-rag-app/data/data.json"

response = urlopen(url)
data = json.load(response)

with open("temp.json", "w") as json_file:
    json.dump(data, json_file)

### Split Documents into Passages

We’ll chunk documents into passages in order to improve the retrieval specificity and to ensure that we can provide multiple passages within the context window of the final question answering prompt.

Here we are chunking documents into 800 token passages with an overlap of 400 tokens.

Here we are using a simple splitter but Langchain offers more advanced splitters to reduce the chance of context being lost.

In [16]:
from langchain.document_loaders import JSONLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter


def metadata_func(record: dict, metadata: dict) -> dict:
    """
    Extracts metadata from a JSON record and populates the metadata dictionary.
    """
    metadata['name'] = record.get('name')
    metadata['summary'] = record.get('summary')
    metadata['url'] = record.get('url')
    metadata['category'] = record.get('category')
    metadata['updated_at'] = record.get('updated_at')
    return metadata


# For more loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/
# And 3rd party loaders https://python.langchain.com/docs/modules/data_connection/document_loaders/#third-party-loaders
loader = JSONLoader(
    file_path="temp.json",
    jq_schema=".[]",
    content_key="content",
    metadata_func=metadata_func,
)

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=800, chunk_overlap=400 #define chunk size and chunk overlap
)
docs = loader.load_and_split(text_splitter=text_splitter)

### Bulk Import Passages

Now that we have split each document into the chunk size of 800, we will now index data to elasticsearch using [ElasticsearchStore.from_documents](https://api.python.langchain.com/en/latest/vectorstores/langchain.vectorstores.elasticsearch.ElasticsearchStore.html#langchain.vectorstores.elasticsearch.ElasticsearchStore.from_documents).

We will use Cloud ID, Password and Index name values set in the `Create cloud deployment` step.

In [18]:
documents = vectorstore.from_documents(
    docs,
    embeddings,
    index_name="langchain_multiquery",
    es_cloud_id=ELASTIC_CLOUD_ID,
    es_api_key=ELASTIC_API_KEY,
)

llm = OpenAI(temperature=0, openai_api_key=OPENAI_API_KEY)

retriever = MultiQueryRetriever.from_llm(vectorstore.as_retriever(), llm)

# Question Answering with MultiQuery Retriever

Now that we have the passages stored in Elasticsearch, we can now ask a question to get the relevant passages.

In [ ]:
from langchain.schema.runnable import RunnableParallel, RunnablePassthrough
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.schema import format_document

import logging

logging.basicConfig()
logging.getLogger("langchain.retrievers.multi_query").setLevel(logging.INFO)

LLM_CONTEXT_PROMPT = ChatPromptTemplate.from_template(
    """You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Be as verbose and educational in your response as possible. 
    
    context: {context}
    Question: "{question}"
    Answer:
    """
)

LLM_DOCUMENT_PROMPT = PromptTemplate.from_template(
    """
---
SOURCE: {name}
{page_content}
---
"""
)


def _combine_documents(
    docs, document_prompt=LLM_DOCUMENT_PROMPT, document_separator="\n\n"
):
    doc_strings = [format_document(doc, document_prompt) for doc in docs]
    return document_separator.join(doc_strings)


_context = RunnableParallel(
    context=retriever | _combine_documents,
    question=RunnablePassthrough(),
)

chain = _context | LLM_CONTEXT_PROMPT | llm

ans = chain.invoke("what is the nasa sales team?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Can you provide information on the sales team at NASA?', '2. How does the sales team at NASA operate?', '3. What are the responsibilities of the sales team at NASA?']


---- Answer ----
The NASA sales team is a part of the Americas region in the sales organization of the company. It is led by two Area Vice-Presidents, Laura Martinez for North America and Gary Johnson for South America. The team is responsible for promoting and selling the company's products and services in the North and South American markets. They work closely with other departments, such as marketing, product development, and customer support, to ensure the company's success in these regions.


**Generate at least two new iteratioins of the previous cells - Be creative.** Did you master Multi-
Query Retriever concepts through this lab?

In [20]:
ans = chain.invoke("what are the documents about?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What is the main topic of the documents?', '2. Can you provide a brief summary of the documents?', '3. What is the general subject matter covered in the documents?']


---- Answer ----

The documents are about various policies and guidelines for employees, including sales strategy, office pet policy, new employee onboarding, intellectual property, work from home updates, and a code of conduct. These documents outline objectives, procedures, and expectations for employees in different areas such as sales, workplace culture, benefits, and professional conduct.


In [21]:
ans = chain.invoke("what are the office pet policies?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are the rules and regulations regarding pets in the workplace?', "2. Can you provide information on the company's policies regarding pets in the office?", '3. How does the company handle pets in the office?']


---- Answer ----
The office pet policy outlines the guidelines and procedures for bringing pets into the workplace. It aims to create a positive and inclusive work environment while ensuring the comfort, safety, and well-being of all employees, visitors, and pets. The policy applies to all employees who wish to bring their pets to the office and covers dogs, cats, and other small, non-exotic animals, subject to approval by the HR department. Employees must obtain prior approval from their supervisor and the HR department before bringing their pets to the office, and the approval process includes submitting a written request, providing proof of vaccinations and any required licenses or permits, and obtaining written consent from all employees who share the workspace with the pet owner. The HR department reserves the right to deny or revoke pet approval based on specific circumstances or concerns raised by other employees. Employees are responsible for the behavior and well-being of thei

In [22]:
ans = chain.invoke("can i bring a dog to the office?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. Is it permissible to have a canine companion in the workplace?', '2. Can I bring my furry friend to the office with me?', '3. Is it acceptable to bring a dog to the office environment?']


---- Answer ----
According to the office pet policy, employees are allowed to bring their pets to the office, including dogs, cats, and other small, non-exotic animals. However, prior approval must be obtained from both the employee's supervisor and the HR department. This approval process includes submitting a written request with a description of the pet, providing proof of vaccinations and any required licenses or permits, and obtaining written consent from all employees who share the workspace with the pet owner. The HR department reserves the right to deny or revoke pet approval based on specific circumstances or concerns raised by other employees. Once approved, employees are responsible for the behavior and well-being of their pets while in the office, and must closely supervise them and promptly address any issues or concerns raised by other staff members. It is also important to note that pets are not allowed in certain areas of the office, such as meeting rooms, restrooms, an

In [ ]:
ans = chain.invoke("im in the office and my dog is sitting under my table, i didnt ask hr before, what can i do?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. How can I handle my dog being in the office with me without asking HR beforehand?', '2. What are some solutions for dealing with my dog sitting under my table at work without prior approval from HR?', "3. In the office, my dog is under my table and I didn't get permission from HR beforehand. What steps can I take to address this situation?"]


---- Answer ----
As per the office pet policy, employees are required to obtain prior approval from their supervisor and the HR department before bringing their pets to the office. If you did not receive approval beforehand, you should immediately remove your dog from the office and follow the pet approval process outlined in the policy. This includes submitting a written request, providing proof of vaccinations and consent from coworkers, and obtaining approval from the HR department. It is important to follow this process to ensure the safety and well-being of all employees, visitors, and pets in the workplace. If you have any further questions or concerns, please reach out to your supervisor or the HR department for guidance.


In [24]:
ans = chain.invoke("best strategy to get a raise in this firm?")

print("---- Answer ----")
print(ans)

INFO:langchain.retrievers.multi_query:Generated queries: ['1. What are some effective tactics for securing a salary increase within this company?', '2. How can one increase their chances of receiving a raise in this particular firm?', '3. What are some proven methods for successfully negotiating a higher salary in this organization?']


---- Answer ----

The best strategy to get a raise in this firm is to consistently demonstrate strong performance, leadership, and a commitment to the company's values. This can be achieved by setting clear, achievable, and measurable performance goals at the beginning of each performance cycle and actively seeking feedback and communication with your supervisor throughout the cycle. Additionally, employees should discuss their career goals and development plans with their supervisor and actively pursue opportunities for advancement within the company. By consistently performing at a high level and showcasing a strong commitment to the company, employees may be considered for promotions or other advancement opportunities, which can lead to a raise. It is also important to regularly review the company's performance management policy and understand the criteria and eligibility for bonuses, awards, or other incentives that may be available.
